In [ ]:
#PCA-GL Projector Knowledge Distillation on Oxford Pets Dataset (smaller dataset)
"""import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import timm 
from tqdm.auto import tqdm 

# ==========================================
# 1. SETUP & REPRODUCIBILITY
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True 

set_seed(42)

# ==========================================
# 2. DATASET (Oxford Pets: 37 Classes, Fast Epochs)
# ==========================================
# Training transforms get augmentations to prevent overfitting over 100 epochs
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

# Test transforms stay perfectly clean
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("Downloading Oxford-IIIT Pet Dataset natively through PyTorch...")
full_trainset = torchvision.datasets.OxfordIIITPet(root='/kaggle/working/data', split='trainval', download=True, transform=train_transform)
full_testset = torchvision.datasets.OxfordIIITPet(root='/kaggle/working/data', split='test', download=True, transform=test_transform)

trainloader = DataLoader(full_trainset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
testloader = DataLoader(full_testset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Loaded {len(full_trainset)} training images and {len(full_testset)} validation images.")

# ==========================================
# 3. ARCHITECTURES & PROJECTORS
# ==========================================
class StudentResNet(nn.Module):
    def __init__(self):
        super().__init__()
        # NO PRETRAINED WEIGHTS - From absolute scratch!
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 37) # 37 classes for Pets

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        feat = self.backbone.layer4(x) 

        out = self.backbone.avgpool(feat)
        out = torch.flatten(out, 1)
        logits = self.backbone.fc(out)
        return feat, logits

class PCAProjector(nn.Module):
    def __init__(self, s_channels, t_dim):
        super().__init__()
        self.query = nn.Conv2d(s_channels, t_dim, 1)
        self.key   = nn.Conv2d(s_channels, t_dim, 1)
        self.value = nn.Conv2d(s_channels, t_dim, 1)

    def forward(self, x):
        q = self.query(x).flatten(2).transpose(1, 2)
        k = self.key(x).flatten(2)
        v = self.value(x).flatten(2).transpose(1, 2)
        attn = torch.softmax(torch.matmul(q, k) / (q.size(-1)**0.5), dim=-1)
        return torch.matmul(attn, v)

class GLProjector(nn.Module):
    def __init__(self, s_channels, t_dim):
        super().__init__()
        self.proj = nn.Conv2d(s_channels, t_dim, 1)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

# ==========================================
# 4. INITIALIZE ALL MODELS
# ==========================================
print("Initializing Models (~21M Teachers vs ~11M Student)...")
teacher_cnn = models.resnet34(weights='IMAGENET1K_V1').to(device).eval()
teacher_vit = timm.create_model('vit_small_patch16_224', pretrained=True).to(device).eval()

set_seed(42); student_base = StudentResNet().to(device)
set_seed(42); student_cnn_kd = StudentResNet().to(device)
set_seed(42); student_vit_kd = StudentResNet().to(device)

pca_proj = PCAProjector(512, 384).to(device)
gl_proj = GLProjector(512, 384).to(device)

# ==========================================
# 5. OPTIMIZERS & SCHEDULERS
# ==========================================
EPOCHS = 100 # Massively increased epochs for from-scratch training
LR = 3e-4

opt_base = optim.AdamW(student_base.parameters(), lr=LR, weight_decay=1e-4)
opt_cnn_kd = optim.AdamW(student_cnn_kd.parameters(), lr=LR, weight_decay=1e-4)
opt_vit_kd = optim.AdamW(list(student_vit_kd.parameters()) + list(pca_proj.parameters()) + list(gl_proj.parameters()), lr=LR, weight_decay=1e-4)

sched_base = optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=EPOCHS)
sched_cnn = optim.lr_scheduler.CosineAnnealingLR(opt_cnn_kd, T_max=EPOCHS)
sched_vit = optim.lr_scheduler.CosineAnnealingLR(opt_vit_kd, T_max=EPOCHS)

# ==========================================
# 6. EVALUATION FUNCTION
# ==========================================
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in testloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, logits = model(imgs)
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return 100 * correct / total

# ==========================================
# 7. MASTER TRAINING LOOP WITH CHECKPOINTING
# ==========================================
best_acc_base = 0.0
best_acc_cnn = 0.0
best_acc_vit = 0.0

print(f"Starting Knowledge Distillation Training for {EPOCHS} Epochs...")

for epoch in range(EPOCHS):
    student_base.train()
    student_cnn_kd.train()
    student_vit_kd.train()
    pca_proj.train()
    gl_proj.train()

    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            x_c = teacher_cnn.conv1(images)
            x_c = teacher_cnn.bn1(x_c)
            x_c = teacher_cnn.relu(x_c)
            x_c = teacher_cnn.maxpool(x_c)
            x_c = teacher_cnn.layer1(x_c)
            x_c = teacher_cnn.layer2(x_c)
            x_c = teacher_cnn.layer3(x_c)
            t_feat_cnn = teacher_cnn.layer4(x_c) 

            t_out_vit = teacher_vit.forward_features(images)
            t_feat_vit = t_out_vit[:, 1:, :] 

        _, s_logits_base = student_base(images)
        loss_base = F.cross_entropy(s_logits_base, labels)
        
        opt_base.zero_grad()
        loss_base.backward()
        opt_base.step()

        s_feat_cnn, s_logits_cnn = student_cnn_kd(images)
        loss_cls_cnn = F.cross_entropy(s_logits_cnn, labels)
        
        s_feat_cnn_norm = F.normalize(s_feat_cnn, p=2, dim=1)
        t_feat_cnn_norm = F.normalize(t_feat_cnn, p=2, dim=1)
        loss_kd_cnn = F.mse_loss(s_feat_cnn_norm, t_feat_cnn_norm) 
        
        loss_cnn_total = loss_cls_cnn + 10.0 * loss_kd_cnn
        
        opt_cnn_kd.zero_grad()
        loss_cnn_total.backward()
        opt_cnn_kd.step()

        s_feat_vit, s_logits_vit = student_vit_kd(images)
        s_feat_resized = F.interpolate(s_feat_vit, size=(14, 14), mode='bilinear', align_corners=False)
        
        s_pca = pca_proj(s_feat_resized)
        s_gl = gl_proj(s_feat_resized)

        loss_cls_vit = F.cross_entropy(s_logits_vit, labels)
        
        s_pca_norm = F.normalize(s_pca, p=2, dim=-1)
        s_gl_norm = F.normalize(s_gl, p=2, dim=-1)
        t_feat_vit_norm = F.normalize(t_feat_vit, p=2, dim=-1)
        
        loss_pca = F.mse_loss(s_pca_norm, t_feat_vit_norm)
        loss_gl = F.mse_loss(s_gl_norm, t_feat_vit_norm)
        
        loss_vit_total = loss_cls_vit + 10.0 * (loss_pca + loss_gl)
        
        opt_vit_kd.zero_grad()
        loss_vit_total.backward()
        opt_vit_kd.step()

    sched_base.step()
    sched_cnn.step()
    sched_vit.step()

    acc_base = evaluate(student_base)
    acc_cnn = evaluate(student_cnn_kd)
    acc_vit = evaluate(student_vit_kd)
    
    print(f"Epoch {epoch+1} | Base: {acc_base:.2f}% | CNN-KD: {acc_cnn:.2f}% | ViT-KD: {acc_vit:.2f}%")

    if acc_base > best_acc_base:
        best_acc_base = acc_base
        torch.save(student_base.state_dict(), '/kaggle/working/best_student_base.pth')
        
    if acc_cnn > best_acc_cnn:
        best_acc_cnn = acc_cnn
        torch.save(student_cnn_kd.state_dict(), '/kaggle/working/best_student_cnn.pth')
        
    if acc_vit > best_acc_vit:
        best_acc_vit = acc_vit
        torch.save(student_vit_kd.state_dict(), '/kaggle/working/best_student_vit.pth')

print("\n==============================================")
print("🎉 Training Complete!")
print(f"Ultimate High Scores -> Baseline: {best_acc_base:.2f}% | CNN-KD: {best_acc_cnn:.2f}% | ViT-KD: {best_acc_vit:.2f}%")
print("Checkpoints successfully saved to /kaggle/working/")
print("==============================================")"""

In [ ]:
#PCA-GL Projector Knowledge Distillation on Imagenette Dataset 
"""import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import timm 
from tqdm.auto import tqdm 

# ==========================================
# 1. SETUP & REPRODUCIBILITY
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True 

set_seed(42)

# ==========================================
# 2. DATASET (Imagenette: High Res, 10 Classes)
# ==========================================
# Augmentations are critical when training from scratch for 100 epochs!
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("Downloading Imagenette natively through PyTorch...")
# PyTorch natively supports Imagenette! Size='full' keeps the high resolution.
full_trainset = torchvision.datasets.Imagenette(root='/kaggle/working/data', split='train', size='full', download=True, transform=train_transform)
full_testset = torchvision.datasets.Imagenette(root='/kaggle/working/data', split='val', size='full', download=True, transform=test_transform)

trainloader = DataLoader(full_trainset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
testloader = DataLoader(full_testset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Loaded {len(full_trainset)} training images and {len(full_testset)} validation images.")

# ==========================================
# 3. ARCHITECTURES & PROJECTORS
# ==========================================
class StudentResNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Training from ABSOLUTE SCRATCH to prove the KD Delta!
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 10) # 10 classes for Imagenette

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        feat = self.backbone.layer4(x) 

        out = self.backbone.avgpool(feat)
        out = torch.flatten(out, 1)
        logits = self.backbone.fc(out)
        return feat, logits

class PCAProjector(nn.Module):
    def __init__(self, s_channels, t_dim):
        super().__init__()
        self.query = nn.Conv2d(s_channels, t_dim, 1)
        self.key   = nn.Conv2d(s_channels, t_dim, 1)
        self.value = nn.Conv2d(s_channels, t_dim, 1)

    def forward(self, x):
        q = self.query(x).flatten(2).transpose(1, 2)
        k = self.key(x).flatten(2)
        v = self.value(x).flatten(2).transpose(1, 2)
        attn = torch.softmax(torch.matmul(q, k) / (q.size(-1)**0.5), dim=-1)
        return torch.matmul(attn, v)

class GLProjector(nn.Module):
    def __init__(self, s_channels, t_dim):
        super().__init__()
        self.proj = nn.Conv2d(s_channels, t_dim, 1)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

# ==========================================
# 4. INITIALIZE ALL MODELS
# ==========================================
print("Initializing Models (~21M Teachers vs ~11M Student)...")
teacher_cnn = models.resnet34(weights='IMAGENET1K_V1').to(device).eval()
teacher_vit = timm.create_model('vit_small_patch16_224', pretrained=True).to(device).eval()

set_seed(42); student_base = StudentResNet().to(device)
set_seed(42); student_cnn_kd = StudentResNet().to(device)
set_seed(42); student_vit_kd = StudentResNet().to(device)

pca_proj = PCAProjector(512, 384).to(device)
gl_proj = GLProjector(512, 384).to(device)

# ==========================================
# 5. OPTIMIZERS & SCHEDULERS
# ==========================================
EPOCHS = 100 
LR = 3e-4

opt_base = optim.AdamW(student_base.parameters(), lr=LR, weight_decay=1e-4)
opt_cnn_kd = optim.AdamW(student_cnn_kd.parameters(), lr=LR, weight_decay=1e-4)
opt_vit_kd = optim.AdamW(list(student_vit_kd.parameters()) + list(pca_proj.parameters()) + list(gl_proj.parameters()), lr=LR, weight_decay=1e-4)

sched_base = optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=EPOCHS)
sched_cnn = optim.lr_scheduler.CosineAnnealingLR(opt_cnn_kd, T_max=EPOCHS)
sched_vit = optim.lr_scheduler.CosineAnnealingLR(opt_vit_kd, T_max=EPOCHS)

# ==========================================
# 6. EVALUATION FUNCTION
# ==========================================
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in testloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, logits = model(imgs)
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return 100 * correct / total

# ==========================================
# 7. MASTER TRAINING LOOP WITH CHECKPOINTING
# ==========================================
best_acc_base = 0.0
best_acc_cnn = 0.0
best_acc_vit = 0.0

print(f"Starting Knowledge Distillation Training for {EPOCHS} Epochs...")

for epoch in range(EPOCHS):
    student_base.train()
    student_cnn_kd.train()
    student_vit_kd.train()
    pca_proj.train()
    gl_proj.train()

    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            x_c = teacher_cnn.conv1(images)
            x_c = teacher_cnn.bn1(x_c)
            x_c = teacher_cnn.relu(x_c)
            x_c = teacher_cnn.maxpool(x_c)
            x_c = teacher_cnn.layer1(x_c)
            x_c = teacher_cnn.layer2(x_c)
            x_c = teacher_cnn.layer3(x_c)
            t_feat_cnn = teacher_cnn.layer4(x_c) 

            t_out_vit = teacher_vit.forward_features(images)
            t_feat_vit = t_out_vit[:, 1:, :] 

        _, s_logits_base = student_base(images)
        loss_base = F.cross_entropy(s_logits_base, labels)
        
        opt_base.zero_grad()
        loss_base.backward()
        opt_base.step()

        s_feat_cnn, s_logits_cnn = student_cnn_kd(images)
        loss_cls_cnn = F.cross_entropy(s_logits_cnn, labels)
        
        s_feat_cnn_norm = F.normalize(s_feat_cnn, p=2, dim=1)
        t_feat_cnn_norm = F.normalize(t_feat_cnn, p=2, dim=1)
        loss_kd_cnn = F.mse_loss(s_feat_cnn_norm, t_feat_cnn_norm) 
        
        loss_cnn_total = loss_cls_cnn + 10.0 * loss_kd_cnn
        
        opt_cnn_kd.zero_grad()
        loss_cnn_total.backward()
        opt_cnn_kd.step()

        s_feat_vit, s_logits_vit = student_vit_kd(images)
        s_feat_resized = F.interpolate(s_feat_vit, size=(14, 14), mode='bilinear', align_corners=False)
        
        s_pca = pca_proj(s_feat_resized)
        s_gl = gl_proj(s_feat_resized)

        loss_cls_vit = F.cross_entropy(s_logits_vit, labels)
        
        s_pca_norm = F.normalize(s_pca, p=2, dim=-1)
        s_gl_norm = F.normalize(s_gl, p=2, dim=-1)
        t_feat_vit_norm = F.normalize(t_feat_vit, p=2, dim=-1)
        
        loss_pca = F.mse_loss(s_pca_norm, t_feat_vit_norm)
        loss_gl = F.mse_loss(s_gl_norm, t_feat_vit_norm)
        
        loss_vit_total = loss_cls_vit + 10.0 * (loss_pca + loss_gl)
        
        opt_vit_kd.zero_grad()
        loss_vit_total.backward()
        opt_vit_kd.step()

    sched_base.step()
    sched_cnn.step()
    sched_vit.step()

    acc_base = evaluate(student_base)
    acc_cnn = evaluate(student_cnn_kd)
    acc_vit = evaluate(student_vit_kd)
    
    print(f"Epoch {epoch+1} | Base: {acc_base:.2f}% | CNN-KD: {acc_cnn:.2f}% | ViT-KD: {acc_vit:.2f}%")

    if acc_base > best_acc_base:
        best_acc_base = acc_base
        torch.save(student_base.state_dict(), '/kaggle/working/best_student_base.pth')
        
    if acc_cnn > best_acc_cnn:
        best_acc_cnn = acc_cnn
        torch.save(student_cnn_kd.state_dict(), '/kaggle/working/best_student_cnn.pth')
        
    if acc_vit > best_acc_vit:
        best_acc_vit = acc_vit
        torch.save(student_vit_kd.state_dict(), '/kaggle/working/best_student_vit.pth')

print("\n==============================================")
print("🎉 Training Complete!")
print(f"Ultimate High Scores -> Baseline: {best_acc_base:.2f}% | CNN-KD: {best_acc_cnn:.2f}% | ViT-KD: {best_acc_vit:.2f}%")
print("Checkpoints successfully saved to /kaggle/working/")
print("==============================================")"""

In [ ]:
#PCA-GL Projector with labels, features and logits all together on Imagenette Dataset

"""import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import timm 
from tqdm.auto import tqdm 

# ==========================================
# 1. SETUP & REPRODUCIBILITY
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True 

set_seed(42)

# ==========================================
# 2. DATASET (WITH RANDAUGMENT)
# ==========================================
# RandAugment is the secret to proving ViT superiority. 
# It makes the task hard enough that the student NEEDS the Transformer's shape bias.
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(num_ops=2, magnitude=12), # Heavy distortion!
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("Downloading Imagenette natively through PyTorch...")
full_trainset = torchvision.datasets.Imagenette(root='/kaggle/working/data', split='train', size='full', download=True, transform=train_transform)
full_testset = torchvision.datasets.Imagenette(root='/kaggle/working/data', split='val', size='full', download=True, transform=test_transform)

trainloader = DataLoader(full_trainset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
testloader = DataLoader(full_testset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# ==========================================
# 3. ARCHITECTURES & PROJECTORS (Liu et al. 2022)
# ==========================================
class StudentResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 10) 

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        feat = self.backbone.layer4(x) 
        out = self.backbone.avgpool(feat)
        out = torch.flatten(out, 1)
        logits = self.backbone.fc(out)
        return feat, logits

class PCAProjector(nn.Module):
    def __init__(self, s_channels, t_dim):
        super().__init__()
        self.query = nn.Conv2d(s_channels, t_dim, 1)
        self.key   = nn.Conv2d(s_channels, t_dim, 1)
        self.value = nn.Conv2d(s_channels, t_dim, 1)

    def forward(self, x):
        q = self.query(x).flatten(2).transpose(1, 2)
        k = self.key(x).flatten(2)
        v = self.value(x).flatten(2).transpose(1, 2)
        attn = torch.softmax(torch.matmul(q, k) / (q.size(-1)**0.5), dim=-1)
        return torch.matmul(attn, v)

class GLProjector(nn.Module):
    def __init__(self, s_channels, t_dim):
        super().__init__()
        self.proj = nn.Conv2d(s_channels, t_dim, 1)

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

def kl_divergence_loss(s_logits, t_logits, temperature=4.0):
    s_soft = F.log_softmax(s_logits / temperature, dim=1)
    t_soft = F.softmax(t_logits / temperature, dim=1)
    return F.kl_div(s_soft, t_soft, reduction='batchmean') * (temperature ** 2)

# ==========================================
# 4. INITIALIZE ALL MODELS
# ==========================================
print("Initializing Models (~21M Teachers vs ~11M Student)...")
teacher_cnn = models.resnet34(weights='IMAGENET1K_V1').to(device).eval()
teacher_vit = timm.create_model('vit_small_patch16_224', pretrained=True).to(device).eval()

set_seed(42); student_base = StudentResNet().to(device)
set_seed(42); student_cnn_kd = StudentResNet().to(device)
set_seed(42); student_vit_kd = StudentResNet().to(device)

pca_proj = PCAProjector(512, 384).to(device)
gl_proj = GLProjector(512, 384).to(device)

# ==========================================
# 5. OPTIMIZERS & SCHEDULERS
# ==========================================
EPOCHS = 100 
LR = 3e-4

opt_base = optim.AdamW(student_base.parameters(), lr=LR, weight_decay=1e-4)
opt_cnn_kd = optim.AdamW(student_cnn_kd.parameters(), lr=LR, weight_decay=1e-4)
opt_vit_kd = optim.AdamW(list(student_vit_kd.parameters()) + list(pca_proj.parameters()) + list(gl_proj.parameters()), lr=LR, weight_decay=1e-4)

sched_base = optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=EPOCHS)
sched_cnn = optim.lr_scheduler.CosineAnnealingLR(opt_cnn_kd, T_max=EPOCHS)
sched_vit = optim.lr_scheduler.CosineAnnealingLR(opt_vit_kd, T_max=EPOCHS)

# ==========================================
# 6. EVALUATION FUNCTION
# ==========================================
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in testloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, logits = model(imgs)
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return 100 * correct / total

# ==========================================
# 7. MASTER TRAINING LOOP (THE PAPER'S EXACT RECIPE)
# ==========================================
best_acc_base, best_acc_cnn, best_acc_vit = 0.0, 0.0, 0.0
IMAGENETTE_CLASSES = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]

# KD Hyperparameters (Classic Hinton 0.1/0.9 split + heavy feature loss)
ALPHA = 0.9 
BETA = 20.0 
TEMP = 4.0

print(f"Starting Cross-Architecture Distillation for {EPOCHS} Epochs...")

for epoch in range(EPOCHS):
    student_base.train()
    student_cnn_kd.train()
    student_vit_kd.train()
    pca_proj.train()
    gl_proj.train()

    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            # CNN Teacher
            x_c = teacher_cnn.conv1(images)
            x_c = teacher_cnn.bn1(x_c)
            x_c = teacher_cnn.relu(x_c)
            x_c = teacher_cnn.maxpool(x_c)
            x_c = teacher_cnn.layer1(x_c)
            x_c = teacher_cnn.layer2(x_c)
            x_c = teacher_cnn.layer3(x_c)
            t_feat_cnn = teacher_cnn.layer4(x_c) 
            
            out_c = teacher_cnn.avgpool(t_feat_cnn)
            out_c = torch.flatten(out_c, 1)
            t_logits_cnn = teacher_cnn.fc(out_c)[:, IMAGENETTE_CLASSES]

            # ViT Teacher
            t_out_vit = teacher_vit.forward_features(images)
            t_feat_vit = t_out_vit[:, 1:, :] 
            t_logits_vit = teacher_vit.forward_head(t_out_vit)[:, IMAGENETTE_CLASSES]

        # --- 1. BASELINE ---
        _, s_logits_base = student_base(images)
        loss_base = F.cross_entropy(s_logits_base, labels)
        
        opt_base.zero_grad()
        loss_base.backward()
        opt_base.step()

        # --- 2. CNN-TAUGHT STUDENT (Homogeneous KD) ---
        s_feat_cnn, s_logits_cnn = student_cnn_kd(images)
        
        loss_ce_cnn = F.cross_entropy(s_logits_cnn, labels)
        loss_kl_cnn = kl_divergence_loss(s_logits_cnn, t_logits_cnn, temperature=TEMP)
        
        s_feat_cnn_norm = F.normalize(s_feat_cnn, p=2, dim=1)
        t_feat_cnn_norm = F.normalize(t_feat_cnn, p=2, dim=1)
        loss_feat_cnn = F.mse_loss(s_feat_cnn_norm, t_feat_cnn_norm) 
        
        # Combined Paper Recipe: (1-Alpha)*CE + Alpha*KL + Beta*Feat
        loss_cnn_total = ((1.0 - ALPHA) * loss_ce_cnn) + (ALPHA * loss_kl_cnn) + (BETA * loss_feat_cnn)
        
        opt_cnn_kd.zero_grad()
        loss_cnn_total.backward()
        opt_cnn_kd.step()

        # --- 3. ViT-TAUGHT STUDENT (Cross-Architecture KD) ---
        s_feat_vit, s_logits_vit = student_vit_kd(images)
        s_feat_resized = F.interpolate(s_feat_vit, size=(14, 14), mode='bilinear', align_corners=False)
        
        s_pca = pca_proj(s_feat_resized)
        s_gl = gl_proj(s_feat_resized)

        loss_ce_vit = F.cross_entropy(s_logits_vit, labels)
        loss_kl_vit = kl_divergence_loss(s_logits_vit, t_logits_vit, temperature=TEMP)
        
        s_pca_norm = F.normalize(s_pca, p=2, dim=-1)
        s_gl_norm = F.normalize(s_gl, p=2, dim=-1)
        t_feat_vit_norm = F.normalize(t_feat_vit, p=2, dim=-1)
        
        loss_feat_vit = F.mse_loss(s_pca_norm, t_feat_vit_norm) + F.mse_loss(s_gl_norm, t_feat_vit_norm)
        
        # Combined Paper Recipe
        loss_vit_total = ((1.0 - ALPHA) * loss_ce_vit) + (ALPHA * loss_kl_vit) + (BETA * loss_feat_vit)
        
        opt_vit_kd.zero_grad()
        loss_vit_total.backward()
        opt_vit_kd.step()

    sched_base.step()
    sched_cnn.step()
    sched_vit.step()

    acc_base = evaluate(student_base)
    acc_cnn = evaluate(student_cnn_kd)
    acc_vit = evaluate(student_vit_kd)
    
    print(f"Epoch {epoch+1} | Base: {acc_base:.2f}% | CNN-KD: {acc_cnn:.2f}% | ViT-KD: {acc_vit:.2f}%")

    if acc_base > best_acc_base:
        best_acc_base = acc_base
        torch.save(student_base.state_dict(), '/kaggle/working/best_student_base.pth')
        
    if acc_cnn > best_acc_cnn:
        best_acc_cnn = acc_cnn
        torch.save(student_cnn_kd.state_dict(), '/kaggle/working/best_student_cnn.pth')
        
    if acc_vit > best_acc_vit:
        best_acc_vit = acc_vit
        torch.save(student_vit_kd.state_dict(), '/kaggle/working/best_student_vit.pth')

print("\n==============================================")
print("🎉 Training Complete!")
print(f"Ultimate High Scores -> Baseline: {best_acc_base:.2f}% | CNN-KD: {best_acc_cnn:.2f}% | ViT-KD: {best_acc_vit:.2f}%")
print("Checkpoints successfully saved to /kaggle/working/")
print("==============================================")"""

In [ ]:
#Without PCA-GL Projector, only logits and labels on Imagenette Dataset
"""import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import timm 
from tqdm.auto import tqdm 

# ==========================================
# 1. SETUP & REPRODUCIBILITY
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True 

set_seed(42)

# ==========================================
# 2. DATASET (WITH RANDAUGMENT)
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(num_ops=2, magnitude=12), # Heavy distortion forces shape reliance!
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("Downloading Imagenette natively through PyTorch...")
full_trainset = torchvision.datasets.Imagenette(root='/kaggle/working/data', split='train', size='full', download=True, transform=train_transform)
full_testset = torchvision.datasets.Imagenette(root='/kaggle/working/data', split='val', size='full', download=True, transform=test_transform)

trainloader = DataLoader(full_trainset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
testloader = DataLoader(full_testset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# ==========================================
# 3. ARCHITECTURE (PURE STUDENT)
# ==========================================
# Without projectors, the student just needs to return its final logits.
class StudentResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 10) 

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        feat = self.backbone.layer4(x) 
        
        out = self.backbone.avgpool(feat)
        out = torch.flatten(out, 1)
        logits = self.backbone.fc(out)
        
        return logits

def kl_divergence_loss(s_logits, t_logits, temperature=4.0):
    s_soft = F.log_softmax(s_logits / temperature, dim=1)
    t_soft = F.softmax(t_logits / temperature, dim=1)
    return F.kl_div(s_soft, t_soft, reduction='batchmean') * (temperature ** 2)

# ==========================================
# 4. INITIALIZE ALL MODELS
# ==========================================
print("Initializing Models (~21M Teachers vs ~11M Student)...")
teacher_cnn = models.resnet34(weights='IMAGENET1K_V1').to(device).eval()
teacher_vit = timm.create_model('vit_small_patch16_224', pretrained=True).to(device).eval()

set_seed(42); student_base = StudentResNet().to(device)
set_seed(42); student_cnn_kd = StudentResNet().to(device)
set_seed(42); student_vit_kd = StudentResNet().to(device)

# ==========================================
# 5. OPTIMIZERS & SCHEDULERS
# ==========================================
EPOCHS = 100 
LR = 3e-4

opt_base = optim.AdamW(student_base.parameters(), lr=LR, weight_decay=1e-4)
opt_cnn_kd = optim.AdamW(student_cnn_kd.parameters(), lr=LR, weight_decay=1e-4)
opt_vit_kd = optim.AdamW(student_vit_kd.parameters(), lr=LR, weight_decay=1e-4)

sched_base = optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=EPOCHS)
sched_cnn = optim.lr_scheduler.CosineAnnealingLR(opt_cnn_kd, T_max=EPOCHS)
sched_vit = optim.lr_scheduler.CosineAnnealingLR(opt_vit_kd, T_max=EPOCHS)

# ==========================================
# 6. EVALUATION FUNCTION
# ==========================================
def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in testloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return 100 * correct / total

# ==========================================
# 7. MASTER TRAINING LOOP (PURE LOGIT DISTILLATION)
# ==========================================
best_acc_base, best_acc_cnn, best_acc_vit = 0.0, 0.0, 0.0
IMAGENETTE_CLASSES = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]

# KD Hyperparameters 
ALPHA = 0.9  # 90% Teacher Logits, 10% True Labels
TEMP = 4.0

print(f"Starting Pure Logit Distillation for {EPOCHS} Epochs...")

for epoch in range(EPOCHS):
    student_base.train()
    student_cnn_kd.train()
    student_vit_kd.train()

    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            # CNN Teacher Logits
            t_logits_cnn_full = teacher_cnn(images)
            t_logits_cnn = t_logits_cnn_full[:, IMAGENETTE_CLASSES]

            # ViT Teacher Logits
            t_logits_vit_full = teacher_vit(images)
            t_logits_vit = t_logits_vit_full[:, IMAGENETTE_CLASSES]

        # --- 1. BASELINE ---
        s_logits_base = student_base(images)
        loss_base = F.cross_entropy(s_logits_base, labels)
        
        opt_base.zero_grad()
        loss_base.backward()
        opt_base.step()

        # --- 2. CNN-TAUGHT STUDENT ---
        s_logits_cnn = student_cnn_kd(images)
        
        loss_ce_cnn = F.cross_entropy(s_logits_cnn, labels)
        loss_kl_cnn = kl_divergence_loss(s_logits_cnn, t_logits_cnn, temperature=TEMP)
        
        # Pure Logit Recipe
        loss_cnn_total = ((1.0 - ALPHA) * loss_ce_cnn) + (ALPHA * loss_kl_cnn)
        
        opt_cnn_kd.zero_grad()
        loss_cnn_total.backward()
        opt_cnn_kd.step()

        # --- 3. ViT-TAUGHT STUDENT ---
        s_logits_vit = student_vit_kd(images)

        loss_ce_vit = F.cross_entropy(s_logits_vit, labels)
        loss_kl_vit = kl_divergence_loss(s_logits_vit, t_logits_vit, temperature=TEMP)
        
        # Pure Logit Recipe
        loss_vit_total = ((1.0 - ALPHA) * loss_ce_vit) + (ALPHA * loss_kl_vit)
        
        opt_vit_kd.zero_grad()
        loss_vit_total.backward()
        opt_vit_kd.step()

    sched_base.step()
    sched_cnn.step()
    sched_vit.step()

    acc_base = evaluate(student_base)
    acc_cnn = evaluate(student_cnn_kd)
    acc_vit = evaluate(student_vit_kd)
    
    print(f"Epoch {epoch+1} | Base: {acc_base:.2f}% | CNN-KD: {acc_cnn:.2f}% | ViT-KD: {acc_vit:.2f}%")

    if acc_base > best_acc_base:
        best_acc_base = acc_base
        torch.save(student_base.state_dict(), '/kaggle/working/best_student_base.pth')
        
    if acc_cnn > best_acc_cnn:
        best_acc_cnn = acc_cnn
        torch.save(student_cnn_kd.state_dict(), '/kaggle/working/best_student_cnn.pth')
        
    if acc_vit > best_acc_vit:
        best_acc_vit = acc_vit
        torch.save(student_vit_kd.state_dict(), '/kaggle/working/best_student_vit.pth')

print("\n==============================================")
print("🎉 Training Complete!")
print(f"Ultimate High Scores -> Baseline: {best_acc_base:.2f}% | CNN-KD: {best_acc_cnn:.2f}% | ViT-KD: {best_acc_vit:.2f}%")
print("Checkpoints successfully saved to /kaggle/working/")
print("==============================================")"""

In [ ]:
#Regularised Projector (PCA + GL)
"""import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import timm 
from tqdm.auto import tqdm 

# ==========================================
# 1. SETUP & REPRODUCIBILITY
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Initializing Final Benchmark Suite on: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False 

set_seed(42)

# ==========================================
# 2. DATASET (Imagenette)
# ==========================================
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(num_ops=2, magnitude=12), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("📥 Downloading Imagenette...")
trainset = torchvision.datasets.Imagenette(root='./data', split='train', size='full', download=True, transform=train_transform)
testset = torchvision.datasets.Imagenette(root='./data', split='val', size='full', download=True, transform=test_transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

# ==========================================
# 3. ARCHITECTURES
# ==========================================

# Standard ResNet-18 for Baseline and CNN-to-CNN KD
class StudentResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 10) 

    def forward(self, x):
        return self.backbone(x)

# The "Final Fix" Student for ViT-to-CNN KD
class RegularizedStudent(nn.Module):
    def __init__(self, vit_embed_dim=384): # vit_small = 384
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 10)
        
        # THE REGULARIZED PROJECTOR (The Inductive Bias Fix)
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout1d(p=0.5) # The "Regularization Parameter"
        self.linear_projector = nn.Linear(512, vit_embed_dim) # Linear to prevent bias-absorption
        self.bn = nn.BatchNorm1d(vit_embed_dim)

    def forward(self, x, return_features=False):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        x = self.backbone.layer3(x)
        feat_map = self.backbone.layer4(x)
        
        pooled = self.pool(feat_map)
        flat = torch.flatten(pooled, 1)
        logits = self.backbone.fc(flat)
        
        if return_features:
            # Dropout forces the BACKBONE to learn robust features, not the projector
            proj = self.dropout(flat)
            proj = self.linear_projector(proj)
            proj = self.bn(proj)
            return logits, proj
            
        return logits

def kl_loss(s_logits, t_logits, T=4.0):
    s_soft = F.log_softmax(s_logits / T, dim=1)
    t_soft = F.softmax(t_logits / T, dim=1)
    return F.kl_div(s_soft, t_soft, reduction='batchmean') * (T**2)

# ==========================================
# 4. INITIALIZATION
# ==========================================
print("Initializing Teachers & Students...")
teacher_cnn = models.resnet34(weights='IMAGENET1K_V1').to(device).eval()
teacher_vit = timm.create_model('vit_small_patch16_224', pretrained=True).to(device).eval()
IMAGENETTE_CLASSES = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]

set_seed(42); student_base = StudentResNet().to(device)
set_seed(42); student_cnn_kd = StudentResNet().to(device)
set_seed(42); student_vit_kd = RegularizedStudent().to(device)

# Optimizers & Schedulers
EPOCHS = 100
LR = 3e-4

opt_base = optim.AdamW(student_base.parameters(), lr=LR, weight_decay=1e-4)
opt_cnn = optim.AdamW(student_cnn_kd.parameters(), lr=LR, weight_decay=1e-4)
opt_vit = optim.AdamW(student_vit_kd.parameters(), lr=LR, weight_decay=1e-4)

sched_base = optim.lr_scheduler.CosineAnnealingLR(opt_base, T_max=EPOCHS)
sched_cnn = optim.lr_scheduler.CosineAnnealingLR(opt_cnn, T_max=EPOCHS)
sched_vit = optim.lr_scheduler.CosineAnnealingLR(opt_vit, T_max=EPOCHS)

# ==========================================
# 5. MASTER TRAINING LOOP
# ==========================================
best_acc_base, best_acc_cnn, best_acc_vit = 0.0, 0.0, 0.0
ALPHA = 0.9  # Logit Weight
GAMMA = 0.1  # Feature Weight (for Student C)
TEMP = 4.0

def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in testloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            logits = model(imgs)
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return 100 * correct / total

print(f"Starting Benchmark Training for {EPOCHS} Epochs...")

for epoch in range(EPOCHS):
    student_base.train(); student_cnn_kd.train(); student_vit_kd.train()
    pbar = tqdm(trainloader, desc=f"Epoch {epoch+1}", leave=False)

    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)

        # 1. Get Teacher Logic
        with torch.no_grad():
            t_logits_cnn = teacher_cnn(images)[:, IMAGENETTE_CLASSES]
            t_feat_vit = teacher_vit.forward_features(images)
            t_cls_vit = t_feat_vit[:, 0]
            t_logits_vit = teacher_vit.forward_head(t_feat_vit)[:, IMAGENETTE_CLASSES]

        # --- STUDENT A: BASELINE (Supervised) ---
        opt_base.zero_grad()
        s_logits_base = student_base(images)
        loss_base = F.cross_entropy(s_logits_base, labels)
        loss_base.backward(); opt_base.step()

        # --- STUDENT B: CNN-TAUGHT (Pure Logit KD) ---
        opt_cnn.zero_grad()
        s_logits_cnn = student_cnn_kd(images)
        loss_ce_cnn = F.cross_entropy(s_logits_cnn, labels)
        loss_kl_cnn = kl_loss(s_logits_cnn, t_logits_cnn, T=TEMP)
        loss_cnn_total = ((1-ALPHA)*loss_ce_cnn) + (ALPHA*loss_kl_cnn)
        loss_cnn_total.backward(); opt_cnn.step()

        # --- STUDENT C: ViT-TAUGHT (Regularized Bottleneck KD) ---
        opt_vit.zero_grad()
        s_logits_vit, s_proj_vit = student_vit_kd(images, return_features=True)
        loss_ce_vit = F.cross_entropy(s_logits_vit, labels)
        loss_kl_vit = kl_loss(s_logits_vit, t_logits_vit, T=TEMP)
        loss_feat_vit = F.mse_loss(s_proj_vit, t_cls_vit)
        # Combine Label + Logit + Regularized Feature Alignment
        loss_vit_total = ((1-ALPHA)*loss_ce_vit) + (ALPHA*loss_kl_vit) + (GAMMA*loss_feat_vit)
        loss_vit_total.backward(); opt_vit.step()

    sched_base.step(); sched_cnn.step(); sched_vit.step()

    # Evaluation
    acc_base = evaluate(student_base)
    acc_cnn = evaluate(student_cnn_kd)
    acc_vit = evaluate(student_vit_kd)
    
    print(f"Epoch {epoch+1} | Base: {acc_base:.2f}% | CNN-KD: {acc_cnn:.2f}% | ViT-KD: {acc_vit:.2f}%")

    if acc_base > best_acc_base:
        best_acc_base = acc_base
        torch.save(student_base.state_dict(), 'best_student_base.pth')
    if acc_cnn > best_acc_cnn:
        best_acc_cnn = acc_cnn
        torch.save(student_cnn_kd.state_dict(), 'best_student_cnn.pth')
    if acc_vit > best_acc_vit:
        best_acc_vit = acc_vit
        torch.save(student_vit_kd.state_dict(), 'best_student_vit_reg.pth')

print("\n" + "="*50)
print(f"🎉 FINAL SCORES -> Base: {best_acc_base:.2f}% | CNN-KD: {best_acc_cnn:.2f}% | ViT-KD: {best_acc_vit:.2f}%")
print("="*50)"""

In [ ]:
#Trying new methods of MGD and Routing - as earlier approaches were not giving desired results
"""import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import timm 
from tqdm.auto import tqdm 

# ==========================================
# 1. SETUP & REPRODUCIBILITY
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Initializing ViT-Taught Showdown on: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False 
set_seed(42)

# ==========================================
# 2. DATASETS (TRAIN, CLEAN TEST, BLUR TEST)
# ==========================================
norm_mean, norm_std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(num_ops=2, magnitude=12), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std)
])

clean_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std)
])

blur_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.GaussianBlur(kernel_size=(15, 15), sigma=(5.0, 5.0)), # Heavy Blur
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std)
])

print("📥 Loading Datasets...")
trainset = torchvision.datasets.Imagenette(root='./data', split='train', size='full', download=True, transform=train_transform)
cleanset = torchvision.datasets.Imagenette(root='./data', split='val', size='full', download=True, transform=clean_transform)
blurset = torchvision.datasets.Imagenette(root='./data', split='val', size='full', download=False, transform=blur_transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
cleanloader = DataLoader(cleanset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
blurloader = DataLoader(blurset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

# ==========================================
# 3. ARCHITECTURES & PROJECTORS
# ==========================================
class StudentResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 10) 

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        feat_layer3 = self.backbone.layer3(x) # For MGD
        feat_layer4 = self.backbone.layer4(feat_layer3) 
        
        pooled = self.backbone.avgpool(feat_layer4)
        flat = torch.flatten(pooled, 1) # For Head Routing
        logits = self.backbone.fc(flat)
        return feat_layer3, flat, logits

# MGD Projector (Paradigm 2)
class MGDProjector(nn.Module):
    def __init__(self, in_channels=256, out_channels=384, mask_ratio=0.5):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.generator = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        )
    def forward(self, x):
        if self.training:
            B, C, H, W = x.shape
            mask = (torch.rand((B, 1, H, W), device=x.device) > self.mask_ratio).float()
            x = x * mask # Apply the blindfold
        return self.generator(x)

def kl_loss(s_logits, t_logits, T=4.0):
    return F.kl_div(F.log_softmax(s_logits/T, dim=1), F.softmax(t_logits/T, dim=1), reduction='batchmean') * (T**2)

# ==========================================
# 4. INITIALIZATION
# ==========================================
print("Initializing ViT Teacher and Students...")
print("-> Loading ViT-Small Teacher (This requires internet)...")
teacher_vit = timm.create_model('vit_small_patch16_224', pretrained=True).to(device).eval()
IMAGENETTE_CLASSES = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]

set_seed(42); student_mgd = StudentResNet().to(device)
set_seed(42); student_routed = StudentResNet().to(device)

mgd_proj = MGDProjector(in_channels=256, out_channels=384, mask_ratio=0.75).to(device)
route_proj = nn.Linear(512, 384).to(device) # Simple linear bottleneck for routing

EPOCHS = 100 
LR = 3e-4

opt_mgd = optim.AdamW(list(student_mgd.parameters()) + list(mgd_proj.parameters()), lr=LR, weight_decay=1e-4)
opt_routed = optim.AdamW(list(student_routed.parameters()) + list(route_proj.parameters()), lr=LR, weight_decay=1e-4)

sched_mgd = optim.lr_scheduler.CosineAnnealingLR(opt_mgd, T_max=EPOCHS)
sched_routed = optim.lr_scheduler.CosineAnnealingLR(opt_routed, T_max=EPOCHS)

# ==========================================
# 5. MASTER TRAINING LOOP
# ==========================================
ALPHA = 0.95  
BETA = 40.0 # MGD MSE Weight
GAMMA = 1.0 # Routing CE Weight
TEMP = 4.0
best_sota_blur = 0.0

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, _, logits = model(imgs)
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return 100 * correct / total

print(f"🔥 Starting ViT-Taught Showdown for {EPOCHS} Epochs...")
print(f"{'Epoch':<6} | {'MGD Clean':<12} | {'MGD Blur':<12}")
print("-" * 65)

for epoch in range(EPOCHS):
    student_mgd.train(); student_routed.train(); mgd_proj.train(); route_proj.train()
    
    for images, labels in tqdm(trainloader, leave=False):
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            # ViT Teacher Forward
            t_out_vit = teacher_vit.forward_features(images) 
            t_feat_spatial = t_out_vit[:, 1:, :] # 196 tokens for MGD
            t_feat_spatial_2d = t_feat_spatial.transpose(1, 2).reshape(images.size(0), 384, 14, 14)
            t_logits = teacher_vit.forward_head(t_out_vit)[:, IMAGENETTE_CLASSES]

        # --- STUDENT 1: MGD (Spatial Masking Paradigm) ---
        opt_mgd.zero_grad()
        s_feat_L3, _, s_logits_mgd = student_mgd(images)
        
        loss_ce_mgd = F.cross_entropy(s_logits_mgd, labels)
        loss_kl_mgd = kl_loss(s_logits_mgd, t_logits, T=TEMP)
        
        s_gen_features = mgd_proj(s_feat_L3) 
        loss_feat_mgd = F.mse_loss(F.normalize(s_gen_features, p=2, dim=1), F.normalize(t_feat_spatial_2d, p=2, dim=1))
        
        loss_mgd_total = ((1-ALPHA)*loss_ce_mgd) + (ALPHA*loss_kl_mgd) + (BETA*loss_feat_mgd)
        loss_mgd_total.backward(); opt_mgd.step()

        # --- STUDENT 2: HEAD ROUTING (Frozen ViT Head Paradigm) ---
        opt_routed.zero_grad()
        _, s_flat_L4, s_logits_routed = student_routed(images)
        
        loss_ce_routed = F.cross_entropy(s_logits_routed, labels)
        loss_kl_routed = kl_loss(s_logits_routed, t_logits, T=TEMP)
        
        # Route the CNN's feature directly into the Teacher's frozen classification head
        s_counterfeit_cls = route_proj(s_flat_L4)
        s_routed_logits = teacher_vit.head(s_counterfeit_cls)[:, IMAGENETTE_CLASSES]
        
        # Force the counterfeit features to produce the correct answers in the ViT Brain
        loss_head_routing = F.cross_entropy(s_routed_logits, labels)
        
        loss_routed_total = ((1-ALPHA)*loss_ce_routed) + (ALPHA*loss_kl_routed) + (GAMMA*loss_head_routing)
        loss_routed_total.backward(); opt_routed.step()

    sched_mgd.step(); sched_routed.step()

    # Instant Evaluation for both Clean and Blur!
    acc_mgd_clean = evaluate(student_mgd, cleanloader)
    acc_mgd_blur = evaluate(student_mgd, blurloader)
    acc_route_clean = evaluate(student_routed, cleanloader)
    acc_route_blur = evaluate(student_routed, blurloader)
    
    print(f"{epoch+1:<6} | {acc_mgd_clean:>10.2f}% | {acc_mgd_blur:>9.2f}% ")

    # Save checkpoints based on Clean accuracy (standard practice)
    if acc_mgd_clean > best_acc_base:  
        best_acc_base = acc_mgd_clean
        torch.save(student_mgd.state_dict(), 'student_mgd_best_clean.pth')
    if acc_mgd_clean >= 90.0 and acc_mgd_blur > best_sota_blur:
        best_sota_blur = acc_mgd_blur
        print(f"🌟 NEW SOTA PEAK! Clean: {acc_mgd_clean:.2f}% | Blur: {acc_mgd_blur:.2f}%")
        torch.save(student_mgd.state_dict(), 'student_mgd_sota_best.pth')"""

In [ ]:
#MGD  - Current model
"""import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader
import torch.nn.functional as F
import random
import numpy as np
import timm 
from tqdm.auto import tqdm 

# ==========================================
# 1. SETUP & REPRODUCIBILITY
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Initializing Extreme MGD Solo Run on: {device}")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False 
set_seed(42)

# ==========================================
# 2. DATASETS (num_workers=0 to prevent Kaggle crash)
# ==========================================
norm_mean, norm_std = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandAugment(num_ops=2, magnitude=12), 
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std)
])

clean_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std)
])

blur_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.GaussianBlur(kernel_size=(15, 15), sigma=(5.0, 5.0)), 
    transforms.ToTensor(),
    transforms.Normalize(norm_mean, norm_std)
])

trainset = torchvision.datasets.Imagenette(root='./data', split='train', size='full', download=True, transform=train_transform)
cleanset = torchvision.datasets.Imagenette(root='./data', split='val', size='full', download=True, transform=clean_transform)
blurset = torchvision.datasets.Imagenette(root='./data', split='val', size='full', download=False, transform=blur_transform)

trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=0, pin_memory=True)
cleanloader = DataLoader(cleanset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)
blurloader = DataLoader(blurset, batch_size=64, shuffle=False, num_workers=0, pin_memory=True)

# ==========================================
# 3. ARCHITECTURES & EXTREME PROJECTOR
# ==========================================
class StudentResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = models.resnet18(weights=None)
        self.backbone.fc = nn.Linear(512, 10) 

    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        x = self.backbone.layer1(x)
        x = self.backbone.layer2(x)
        feat_layer3 = self.backbone.layer3(x) 
        feat_layer4 = self.backbone.layer4(feat_layer3) 
        
        pooled = self.backbone.avgpool(feat_layer4)
        flat = torch.flatten(pooled, 1) 
        logits = self.backbone.fc(flat)
        return feat_layer3, logits

class MGDProjector(nn.Module):
    def __init__(self, in_channels=256, out_channels=384, mask_ratio=0.75): # 75% BLINDFOLD
        super().__init__()
        self.mask_ratio = mask_ratio
        self.generator = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        )
    def forward(self, x):
        if self.training:
            B, C, H, W = x.shape
            mask = (torch.rand((B, 1, H, W), device=x.device) > self.mask_ratio).float()
            x = x * mask 
        return self.generator(x)

def kl_loss(s_logits, t_logits, T=4.0):
    return F.kl_div(F.log_softmax(s_logits/T, dim=1), F.softmax(t_logits/T, dim=1), reduction='batchmean') * (T**2)

# ==========================================
# 4. INITIALIZATION (With HF Token bypass if needed)
# ==========================================
print("-> Loading ViT-Small Teacher...")
try:
    teacher_vit = timm.create_model('vit_small_patch16_224', pretrained=True).to(device).eval()
except Exception as e:
    print("HF Hub failed. Ensure your HF_TOKEN is set or you are authenticated.")
    raise e

IMAGENETTE_CLASSES = [0, 217, 482, 491, 497, 566, 569, 571, 574, 701]

set_seed(42); student_mgd = StudentResNet().to(device)
mgd_proj = MGDProjector(in_channels=256, out_channels=384, mask_ratio=0.75).to(device)

EPOCHS = 100 
LR = 3e-4

opt_mgd = optim.AdamW(list(student_mgd.parameters()) + list(mgd_proj.parameters()), lr=LR, weight_decay=1e-4)
sched_mgd = optim.lr_scheduler.CosineAnnealingLR(opt_mgd, T_max=EPOCHS)

# ==========================================
# 5. MASTER TRAINING LOOP (Extreme Settings)
# ==========================================
ALPHA = 0.95 # 95% reliance on ViT Teacher's logits
BETA = 40.0  # Double penalty for failing to reconstruct shape
TEMP = 4.0

def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            _, logits = model(imgs) # Unpack only 2 items now
            _, predicted = torch.max(logits, 1)
            total += lbls.size(0)
            correct += (predicted == lbls).sum().item()
    return 100 * correct / total

print(f"🔥 Starting Extreme MGD Run for {EPOCHS} Epochs...")
print(f"{'Epoch':<6} | {'MGD Clean':<12} | {'MGD Blur':<12}")
print("-" * 35)

best_acc_base = 0.0
best_sota_blur = 0.0 # Tracks the highest blur achieved

for epoch in range(EPOCHS):
    student_mgd.train(); mgd_proj.train()
    
    for images, labels in tqdm(trainloader, leave=False):
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            t_out_vit = teacher_vit.forward_features(images) 
            t_feat_spatial = t_out_vit[:, 1:, :] 
            t_feat_spatial_2d = t_feat_spatial.transpose(1, 2).reshape(images.size(0), 384, 14, 14)
            t_logits = teacher_vit.forward_head(t_out_vit)[:, IMAGENETTE_CLASSES]

        opt_mgd.zero_grad()
        s_feat_L3, s_logits_mgd = student_mgd(images)
        
        loss_ce_mgd = F.cross_entropy(s_logits_mgd, labels)
        loss_kl_mgd = kl_loss(s_logits_mgd, t_logits, T=TEMP)
        
        s_gen_features = mgd_proj(s_feat_L3) 
        loss_feat_mgd = F.mse_loss(F.normalize(s_gen_features, p=2, dim=1), F.normalize(t_feat_spatial_2d, p=2, dim=1))
        
        loss_mgd_total = ((1-ALPHA)*loss_ce_mgd) + (ALPHA*loss_kl_mgd) + (BETA*loss_feat_mgd)
        loss_mgd_total.backward(); opt_mgd.step()

    sched_mgd.step()

    acc_mgd_clean = evaluate(student_mgd, cleanloader)
    acc_mgd_blur = evaluate(student_mgd, blurloader)
    
    print(f"{epoch+1:<6} | {acc_mgd_clean:>10.2f}% | {acc_mgd_blur:>9.2f}%")

    # 1. Standard Best Clean Saver
    if acc_mgd_clean > best_acc_base:  
        best_acc_base = acc_mgd_clean
        torch.save(student_mgd.state_dict(), 'student_mgd_best_clean.pth')

    # 2. DYNAMIC SOTA SAVER
    if acc_mgd_clean >= 90.0 and acc_mgd_blur > best_sota_blur:
        best_sota_blur = acc_mgd_blur
        print(f"🌟 NEW SOTA PEAK! Saving model with Blur: {acc_mgd_blur:.2f}%")
        torch.save(student_mgd.state_dict(), 'student_mgd_sota_best.pth')"""